# 04 — Supervised LSTM

Train a two-layer LSTM on a window of features to predict each name's
**cross-sectional rank** of forward 5-day return, then hold the top quartile.

**Why rank, not return.** Most of the variance of an absolute forward return is
the market factor — nearly unforecastable, and for a long-only book with no
index hedge, unusable even when forecast correctly. The network would spend its
capacity on the one component it cannot act on. Ranking each name against the
rest of the universe on the same date leaves the idiosyncratic part, which is
what choosing between stocks can actually monetize.

**Why the split is by date.** Every symbol's training window ends before any
symbol's validation window begins. Splitting by row after stacking symbols would
put one name's future alongside another's past.

Standalone: no `portfolio_agent` import. Needs `torch`.

## Setup

In [ ]:
# Dependencies. Torch is only needed by the two learned strategies (04, 05).
# !pip install -q pandas numpy pyarrow matplotlib huggingface_hub torch

import sys, pathlib

# afa_lab.py sits next to this notebook. On Colab (or anywhere the file is
# missing) fetch it from the repo — that is the only network call that touches
# GitHub, and nothing else here imports the portfolio_agent package.
if not pathlib.Path("afa_lab.py").exists():
    import urllib.request
    URL = ("https://raw.githubusercontent.com/3dwag98/afa/main/"
           "notebooks/standalone/afa_lab.py")
    urllib.request.urlretrieve(URL, "afa_lab.py")
    print("fetched afa_lab.py")

sys.path.insert(0, ".")
import afa_lab as L

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("toolkit loaded | torch available:", L.TORCH_AVAILABLE)

In [ ]:
# NSE large caps. Any symbol absent from the dataset is skipped rather than
# failing the run, so this list does not have to be exactly right.
UNIVERSE = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
    "HINDUNILVR",
    "ITC",
    "SBIN",
    "BHARTIARTL",
    "KOTAKBANK",
    "LT",
    "AXISBANK",
    "ASIANPAINT",
    "MARUTI",
    "SUNPHARMA",
    "TITAN",
    "ULTRACEMCO",
    "WIPRO",
    "NESTLEIND",
    "BAJFINANCE",
    "TATAMOTORS",
    "TATASTEEL",
    "POWERGRID",
    "NTPC",
    "ONGC",
    "HCLTECH",
    "JSWSTEEL",
    "GRASIM",
    "CIPLA",
    "COALINDIA"
]

START_DATE = "2018-01-01"
END_DATE   = None          # None = up to the dataset's last session
CACHE      = "data_cache"  # downloaded parquet files land here and are reused

print(len(UNIVERSE), "symbols requested")

In [ ]:
# Ingestion. One small parquet per symbol is pulled from the Hub dataset
# `vishnun0027/indian-market-historical-ohlcv` (2,421 NSE/BSE equities) and
# cleaned. Downloading per symbol rather than snapshotting the repo means a
# 30-name universe fetches 30 small files instead of 283 MB.
#
# Cleaning, in order: back-adjust OHLC by adj_close/close so a split is not read
# as a 90% crash, coerce numerics, drop unparseable dates and missing closes
# (rather than forward-filling, so a gap stays visible), and drop duplicate
# sessions keeping the last.
#
# If the Hub is unreachable the toolkit falls back to a synthetic panel and says
# so loudly. Synthetic results describe the generator, not the market.

panel = L.load_panel(UNIVERSE, start_date=START_DATE, end_date=END_DATE,
                     cache_dir=CACHE)

close = L.align_close_matrix(panel)
print(f"{len(panel)} symbols | {close.index.min().date()} -> {close.index.max().date()}"
      f" | {len(close)} sessions")

In [ ]:
# Features are computed per symbol and left NaN until each window has filled.
# They are never back-filled: a back-filled indicator is a look-ahead, and it is
# invisible in every metric downstream.
feature_panel = L.build_feature_panel(panel)

sample = feature_panel[sorted(feature_panel)[0]]
print(f"{len(sample.columns)} features:", list(sample.columns))
display(sample.dropna().tail(3))

In [ ]:
# Simulation settings, shared by every notebook so the strategies are comparable.
#
# execution_lag=1 is the property that keeps this honest: a signal computed from
# day t's close is traded into day t+1's return. The engine refuses lag=0.
config = L.BacktestConfig(
    initial_capital=1_000_000.0,
    cost_bps=25.0,        # all-in round trip for Indian cash equities
    max_weight=0.10,
    rebalance_days=5,     # weekly; the main control over turnover
    max_gross=1.0,        # long-only, unlevered
    execution_lag=1,
)

benchmark = L.equal_weight_benchmark(close, config)
print("equal-weight buy & hold:",
      {k: round(v, 4) for k, v in benchmark.stats.items()
       if k in ("cagr", "sharpe", "max_drawdown")})

## Build the supervised panel

The standardizer is fitted on **training rows only**. Fitting it on the whole
history leaks the test period's moments into the transform — a small leak, and
entirely invisible in the resulting metrics, which is what makes it worth being
strict about.

In [ ]:
supervised = L.build_supervised_panel(
    feature_panel, close,
    sequence_length=30,     # sessions of history per sample
    horizon=5,              # forward return being ranked
    train_fraction=0.70,
    val_fraction=0.15,
)

for split in ("train", "val", "test"):
    X = supervised[f"X_{split}"]
    print(f"{split:5s} {X.shape[0]:6d} windows  shape {X.shape[1:]}")
print(f"\ntrain ends {supervised['train_end'].date()} | "
      f"validation ends {supervised['val_end'].date()}")
print("features:", supervised["features"])

## Train

Early stopping is on validation loss and the returned weights are the **best**
epoch's, not the last. At this signal-to-noise ratio a model reliably keeps
improving in-sample long after it has stopped generalizing.

Rank IC (Spearman correlation between predicted and realized ordering) is the
metric that matches how the output is used — the predictions become a ranking,
so monotone agreement is what matters, not squared error. Values around 0.02–0.05
are normal and useful in equity forecasting; anything above ~0.15 on daily data
usually means a leak.

In [ ]:
trained = L.train_lstm(
    supervised,
    epochs=40,
    batch_size=256,
    learning_rate=1e-3,
    hidden_size=64,
    patience=6,
    device="auto",
    seed=42,
)

print(f"\nbest epoch {trained['best_epoch']} | val loss {trained['best_val_loss']:.5f}")
L.plot_training_curve(trained["history"],
                      ["train_loss", "val_loss", "val_rank_ic"],
                      title="LSTM training")

In [ ]:
# Held-out rank IC — the number that decides whether anything below is meaningful.
import torch

model, device = trained["model"], trained["device"]
X_test = torch.tensor(supervised["X_test"]).to(device)
y_test = torch.tensor(supervised["y_test"]).to(device)

with torch.no_grad():
    predictions = model(X_test)

print(f"test rank IC: {L._rank_ic(predictions, y_test):+.4f}")
print(f"test MSE    : {float(torch.nn.functional.mse_loss(predictions, y_test)):.5f}")
print("\nA rank IC near zero means the ranking below is noise, and the equity")
print("curve is measuring the sample rather than the model.")

## Signal & backtest

Predictions are produced for every date, including the training period, so the
equity curve can be split. **Only the segment after the validation cut is
evidence of anything** — the part before it is the model recalling its own
training data.

In [ ]:
lstm_scores = L.lstm_scores(trained, feature_panel, close, top_fraction=0.25)

## Backtest

Against equal-weight buy-and-hold of the same names — the honest comparison for a long-only stock picker. Beating cash is not the question.

In [ ]:
result = L.run_backtest(lstm_scores, close, config)

comparison = L.compare_stats({"lstm": result, "equal weight": benchmark})
display(comparison)

## Analysis

In [ ]:
L.plot_equity({"lstm": result}, title="lstm vs equal weight",
              benchmark=benchmark.returns)

In [ ]:
L.plot_return_profile(result, "lstm")

In [ ]:
L.plot_exposure(result, "lstm")

In [ ]:
L.plot_weight_heatmap(result, title="lstm: allocation over time")

In [ ]:
# In-sample versus out-of-sample, split at the validation cut. A large gap here
# is the normal outcome and the reason the full-period curve is not the result.
cut = supervised["val_end"]

oos_scores = lstm_scores[lstm_scores.index > cut]
oos_close = close[close.index > cut]
oos = L.run_backtest(oos_scores, oos_close, config)
oos_benchmark = L.equal_weight_benchmark(oos_close, config)

is_scores = lstm_scores[lstm_scores.index <= cut]
is_close = close[close.index <= cut]
in_sample = L.run_backtest(is_scores, is_close, config)

display(L.compare_stats({
    "in-sample (not evidence)": in_sample,
    "out-of-sample": oos,
    "equal weight (oos)": oos_benchmark,
}))

L.plot_equity({"out-of-sample": oos}, benchmark=oos_benchmark.returns,
              title=f"LSTM out-of-sample (from {cut.date()})", log_scale=False)

---

## What this does and does not show

Read before quoting any number above.

- **Survivorship.** The universe is today's large caps, applied to history. Names
  that were large caps in 2018 and are not now are absent, and they are absent
  precisely because they did badly. Every long-only result here is biased upward
  by an amount this notebook cannot measure. A point-in-time constituent list is
  the only fix, and this dataset does not carry one.
- **One universe, one period.** Thirty names over a few years is a single draw.
  The difference between two strategies here is well within what the draw alone
  could produce.
- **Costs are a flat 25 bps.** Real cost scales with size and with how illiquid
  the name is, and the fill is assumed at the close. A strategy whose edge is
  this side of costs is not distinguishable from one that has no edge.
- **No point-in-time fundamentals, no corporate actions beyond the price
  adjustment**, and no circuit-limit modelling. On Indian equities a
  circuit-locked session is untradeable, and the simulation will happily trade it.
- **Parameters were chosen, not fitted.** Nothing here is tuned on a held-out
  period. That is deliberate — tuning on this sample and reporting the result
  would be reporting the tuning.

The purpose of these notebooks is to make the mechanism legible and modifiable,
not to establish that any of these strategies makes money.